# Sleep Staging with HMMs: Dataset Construction (NBK 1)

This notebook builds the feature dataset using raw Sleep-EDF Expanded recordings. It pairs each polysomnogram with its hypnogram, maps the older sleep-stage labels to W, N1, N2, N3, and REM, cuts each recording into 30-second epochs, and trims it to the main sleep period. It then computes log band power from the EEG and EOG, z-scores the features within each recording, and writes `features.csv.gz`, `stage_counts.csv`, and `features_lights_off.csv.gz` (for a sensitivity check), which are used in downstream analysis.

**Dataset:** We use the [Sleep-EDF Database Expanded v1.0.0](https://doi.org/10.13026/C2X676) from PhysioNet [1], which contains 197 recordings from two studies. We specifically use only the **Sleep Cassette (SC)** study: 153 recordings from 78 healthy subjects aged 25–101, made in 1987–1991 to study the effect of age on sleep. Subjects wore a modified Walkman-like cassette recorder at home for two day-night periods in a row, about 20 hours each. We leave out the Sleep Telemetry (ST) study because it tested a sleep drug (temazepam), which could change the sleep patterns we want to model.

**Files:** Each recording has two files:
- The **PSG** (polysomnogram, `*-PSG.edf`) holds the signals measured during the recording. These are the observations.
- The **hypnogram** (`*-Hypnogram.edf`) holds the sleep stage that a trained technician assigned to each part of the recording. These are the expert labels.

**Signals:** We use two of the PSG channels, both sampled at 100 Hz:
- **EEG** (electroencephalogram) measures the brain's electrical activity through electrodes on the scalp. We use the Fpz-Cz channel, which runs from the front to the top of the head.
- **EOG** (electrooculogram) measures eye movements through electrodes near the eyes. It may help separate REM from N1, which look similar in the EEG.

**Labels:** The hypnograms use the older Rechtschaffen and Kales (R&K) labels, which we map to the modern five stages:

| R&K label | Our label | Meaning |
|---|---|---|
| W | W | Awake |
| 1 | N1 | Lightest sleep, a brief transition between wake and sleep |
| 2 | N2 | Light sleep, usually the largest share of the night |
| 3 and 4 | N3 | Deep (slow-wave) sleep |
| R | REM | Rapid eye movement sleep, linked to vivid dreaming |
| Movement time, ? | unscored | Dropped, and the sequence is split there |

Note that REM is not Stage 4: in R&K scoring, Stage 4 was the deepest non-REM stage.

**Trimming:** The recordings run all day. As such, most of each one is daytime wake, which would dominate the model. We trim in three steps:
1. Split the sleep epochs into blocks wherever there is a gap of three hours or more without sleep.
2. Take the block with the most sleep epochs as the main sleep period.
3. Keep that block plus 30 minutes before and after it.

We use a three-hour gap because shorter gaps drop more sleep, as the check on the trimming rule below shows. Sleep onset and offset come from the expert hypnograms for every recording, including held-out subjects. The stage labels of held-out subjects are never used to fit the models or match states to stages. As a robustness check, we repeat the K = 5 experiment with a window based on recorded lights-off times, which uses no stage labels to set the window.

**References:**

[1] B. Kemp, A. H. Zwinderman, B. Tuk, H. A. C. Kamphuisen, and J. J. L. Oberyé, "Analysis of a sleep-dependent neuronal feedback loop: the slow-wave microcontinuity of the EEG," *IEEE Transactions on Biomedical Engineering*, vol. 47, no. 9, pp. 1185–1194, 2000. [doi:10.1109/10.867928](https://doi.org/10.1109/10.867928)

[2] R. Vallat and M. P. Walker, "An open-source, high-performance tool for automated sleep staging," *eLife*, vol. 10, p. e70092, 2021. [doi:10.7554/eLife.70092](https://doi.org/10.7554/eLife.70092)

---

### Import statements

In [55]:
import os
import numpy as np
import pandas as pd
import mne
from scipy.signal import welch
import helpers

---

### Pairing polysomnogram (PSG) and hypnogram files

Each recording has a polysomnogram (PSG) file and a hypnogram file that share the same first six characters, such as `SC4001E0-PSG.edf` and `SC4001EC-Hypnogram.edf`. We pair them on those six characters and check that every PSG has exactly one hypnogram. In `SC4001`, the two digits after `SC4` are the **subject** (`00`) and the next digit is the **night** (`1`).

`SC4001` and `SC4002` are two nights from the same person, so splitting by recording could put one night in training and the other in testing. We keep the subject as its own column so that **both nights always stay together**.

In [56]:
rows = []
for psg in sorted(f for f in os.listdir(helpers.RAW) if f.endswith("-PSG.edf")):
    recording = psg[:6]
    hyps = [f for f in os.listdir(helpers.RAW)
            if f.startswith(recording) and f.endswith("-Hypnogram.edf")]
    assert len(hyps) == 1, f"{recording}: found {len(hyps)} hypnograms"
    rows.append({"recording": recording, "subject": recording[3:5], "night": int(recording[5]),
                 "psg_path": os.path.join(helpers.RAW, psg), "hyp_path": os.path.join(helpers.RAW, hyps[0])})

files = pd.DataFrame(rows)
nights = files.groupby("subject").size()
print(f"{len(files)} recordings, {len(nights)} subjects")
print("Subjects with one night:", nights[nights == 1].index.tolist())
files[["recording", "subject", "night"]].head()

153 recordings, 78 subjects
Subjects with one night: ['13', '36', '52']


,recording,subject,night
0,SC4001,00,1
1,SC4002,00,2
2,SC4011,01,1
3,SC4012,01,2
4,SC4021,02,1


All 153 recordings are paired, covering 78 subjects. Most subjects have two nights. Subjects 13, 36, and 52 have only one because the other recording was lost to a failing cassette or laserdisk.

---

### Mapping the hypnogram labels

A hypnogram is a list of annotations, each with a start time, a duration, and an R&K label. We map the labels to five stages, merging stages 3 and 4 into N3. Movement time and `?` carry no stage, so they become `unscored`. We do not guess a stage for them: they are dropped later, and the sequence is split there so that the HMM never counts a transition across the gap.

We also check that every annotation starts and lasts a multiple of 30 seconds. If that holds, each 30-second epoch lies inside exactly one annotation and its label is never ambiguous.

In [57]:
STAGE_MAP = {
    "Sleep stage W": "W",
    "Sleep stage 1": "N1",
    "Sleep stage 2": "N2",
    "Sleep stage 3": "N3",
    "Sleep stage 4": "N3",
    "Sleep stage R": "REM",
}

rows = []
for path in files["hyp_path"]:
    for a in mne.read_annotations(path):
        rows.append({"description": a["description"], "onset": a["onset"],
                     "duration": a["duration"]})
ann = pd.DataFrame(rows)

assert (ann["onset"] % helpers.EPOCH_SEC == 0).all()
assert (ann["duration"] % helpers.EPOCH_SEC == 0).all()

ann["stage"] = ann["description"].map(STAGE_MAP).fillna("unscored")
ann["epochs"] = (ann["duration"] / helpers.EPOCH_SEC).astype(int)
ann.groupby(["description", "stage"])["epochs"].sum()

description    stage   
Movement time  unscored       128
Sleep stage 1  N1           21522
Sleep stage 2  N2           69132
Sleep stage 3  N3            8793
Sleep stage 4  N3            4246
Sleep stage ?  unscored     25047
Sleep stage R  REM          25835
Sleep stage W  W           285937
Name: epochs, dtype: int64

All annotations line up with the 30-second grid. N2 is the most common sleep stage (69,132 epochs), and N3 combines 8,793 epochs of stage 3 with 4,246 of stage 4. Wake (285,937 epochs) is more than twice all sleep stages combined, which is why we trim. These counts cover the full hypnograms. Epochs past the end of the PSG are cut when we build the epochs, so the wake and unscored counts are lower from then on.

---

### Cutting recordings into 30-second epochs

Sleep is scored in 30-second epochs, so the EEG and EOG are cut in the same way. At 100 Hz, each epoch is 3,000 samples. Because every annotation starts on the 30-second grid, dividing its start time by 30 gives the exact epoch where it begins. Epochs that no scored annotation covers stay `unscored`.

In [58]:
def load_epochs(row):
    raw = mne.io.read_raw_edf(row.psg_path, verbose="error",
                              include=[helpers.EEG_CHANNEL, helpers.EOG_CHANNEL])
    sfreq = int(raw.info["sfreq"])
    samples = helpers.EPOCH_SEC * sfreq
    n_epochs = raw.n_times // samples

    signals = {}
    for channel in (helpers.EEG_CHANNEL, helpers.EOG_CHANNEL):
        x = raw.get_data(picks=[channel])[0, :n_epochs * samples]
        signals[channel] = x.reshape(n_epochs, samples)

    stages = np.full(n_epochs, "unscored", dtype=object)
    for a in mne.read_annotations(row.hyp_path):
        start = int(a["onset"] // helpers.EPOCH_SEC)
        stop  = start + int(a["duration"] // helpers.EPOCH_SEC)
        stages[start:stop] = STAGE_MAP.get(a["description"], "unscored")

    return signals, stages, sfreq

signals, stages, sfreq = load_epochs(files.iloc[0])
print(files.iloc[0]["recording"], "at", sfreq, "Hz")
print({channel: x.shape for channel, x in signals.items()})
print("View signals:", helpers.lightwave_url(files.iloc[0]["psg_path"]))
pd.Series(stages).value_counts()

SC4001 at 100 Hz
{'EEG Fpz-Cz': (2650, 3000), 'EOG horizontal': (2650, 3000)}
View signals: https://www.physionet.org/lightwave/?db=sleep-edfx/1.0.0&record=sleep-cassette/SC4001E0-PSG.edf&t0=00:00:00


W      1997
N2      250
N3      220
REM     125
N1       58
Name: count, dtype: int64

The first recording, SC4001, has 2,650 epochs per channel, or about 22 hours. Of these, 1,997 are wake and 653 are sleep. The link opens the same recording in PhysioNet's viewer, which shows the signals but not the stages.

---

### Trimming to the main sleep period

Left untrimmed, wake would dominate both Baum–Welch and the metrics. We split a recording's sleep epochs into blocks wherever there are three hours or more without sleep, keep the block with the most sleep, and add 30 minutes of wake on each side. The 30 minutes keeps some falling-asleep and waking-up transitions for the HMM to learn from. Keeping only the main block leaves out daytime sleep, so our results describe the main sleep period.

In [59]:
WAKE_MARGIN = 30 * 60 // helpers.EPOCH_SEC
MAX_GAP = 3 * 60 * 60 // helpers.EPOCH_SEC

def sleep_blocks(stages, max_gap=MAX_GAP):
    asleep = np.flatnonzero(np.isin(stages, ["N1", "N2", "N3", "REM"]))
    return np.split(asleep, np.flatnonzero(np.diff(asleep) > max_gap) + 1)

def trim_bounds(stages, max_gap=MAX_GAP):
    main = max(sleep_blocks(stages, max_gap), key=len)
    start = max(0, main[0] - WAKE_MARGIN)
    stop  = min(len(stages), main[-1] + 1 + WAKE_MARGIN)
    return start, stop

start, stop = trim_bounds(stages)
print(f"Keeping epochs {start} to {stop} of {len(stages)} "
      f"({(stop - start) * helpers.EPOCH_SEC / 3600:.1f} of "
      f"{len(stages) * helpers.EPOCH_SEC / 3600:.1f} hours)")
pd.DataFrame({"before": pd.Series(stages).value_counts(),
              "after":  pd.Series(stages[start:stop]).value_counts()}).fillna(0).astype(int)

Keeping epochs 961 to 1802 of 2650 (7.0 of 22.1 hours)


,before,after
N1,58,58
N2,250,250
N3,220,220
REM,125,125
W,1997,188


For SC4001 the rule keeps 7.0 of 22.1 hours. Wake drops from 1,997 to 188 epochs, and no sleep epochs are lost.

---

### Computing log band power

Now, we summarize each epoch by its power in standard EEG frequency bands: delta, theta, alpha, sigma, and beta. Our bands follow [YASA](https://github.com/raphaelvallat/yasa/blob/master/src/yasa/staging.py), a published sleep-staging tool [2], except that YASA splits delta into two bands and starts it at 0.4 Hz. We estimate the power spectrum with Welch's method: each epoch is cut into overlapping 4-second pieces, and their spectra are averaged to reduce noise (resolution 0.25 Hz). We then add up the power in each band and take the log. Power is never negative and has a long right tail, and the log brings it closer to the Gaussian that our HMM assumes.

Eye movements are slow, so we summarize the EOG with two low-frequency bands (0.5–2 and 2–5 Hz). There is no equally standard set of bands for the EOG, so these are our own choice, meant as a simple summary of eye movement. The recordings were already filtered to remove activity below 0.5 Hz, so no band starts below that.

In [60]:
EEG_BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 12),
             "sigma": (12, 16), "beta": (16, 30)}
EOG_BANDS = {"eog_slow": (0.5, 2), "eog_fast": (2, 5)}
FEATURES = list(EEG_BANDS) + list(EOG_BANDS)

def log_band_power(epochs, sfreq, bands):
    freqs, psd = welch(epochs, fs=sfreq, window="hann",
                       nperseg=4 * sfreq, noverlap=2 * sfreq)
    width = freqs[1] - freqs[0]
    power = {}
    for name, (low, high) in bands.items():
        in_band = (freqs >= low) & (freqs < high)
        power[name] = np.log(psd[:, in_band].sum(axis=1) * width)
    return pd.DataFrame(power)

def compute_features(signals, sfreq):
    return pd.concat([log_band_power(signals[helpers.EEG_CHANNEL], sfreq, EEG_BANDS),
                      log_band_power(signals[helpers.EOG_CHANNEL], sfreq, EOG_BANDS)], axis=1)

example = compute_features({c: x[start:stop] for c, x in signals.items()}, sfreq)
example.groupby(stages[start:stop]).mean().round(2)

,delta,theta,alpha,sigma,beta,eog_slow,eog_fast
N1,-23.45,-24.72,-25.91,-27.18,-26.64,-22.70,-24.71
N2,-22.31,-24.40,-26.00,-26.18,-27.26,-23.48,-24.64
N3,-20.90,-23.90,-25.56,-26.34,-27.49,-22.42,-23.83
REM,-23.57,-24.83,-26.15,-27.48,-27.07,-22.09,-24.40
W,-22.12,-24.34,-25.68,-26.83,-25.93,-20.51,-22.90


The values are negative because the signals are in volts, so the power is far below 1. In SC4001, N3 has the most delta, N2 the most sigma (only just ahead of N3), and wake the most beta and EOG power. N1 and REM have nearly the same delta and theta, but differ more in beta and slow EOG power. One surprise is alpha, which is slightly higher in N3 than in wake. That said, this is a single recording before normalization.

---

### Normalizing within each recording

Absolute EEG power differs a lot between people and between nights for reasons unrelated to sleep stage, such as electrode contact. We therefore z-score each feature using only its own recording. This happens after trimming, so that a night with more daytime wake is not shifted relative to the others, and after dropping unscored epochs. 

No recording's scaling depends on any other recording, so nothing passes from training to test subjects.

In [61]:
tables, counts, all_stages = [], [], {}
for row in files.itertuples():
    signals, stages, sfreq = load_epochs(row)
    start, stop = trim_bounds(stages)
    all_stages[row.recording] = stages

    table = compute_features({c: x[start:stop] for c, x in signals.items()}, sfreq)
    table.insert(0, "recording", row.recording)
    table.insert(1, "subject", row.subject)
    table.insert(2, "night", row.night)
    table.insert(3, "epoch", np.arange(start, stop))
    table.insert(4, "stage", stages[start:stop])

    table = table[table["stage"] != "unscored"].copy()
    table[FEATURES] = (table[FEATURES] - table[FEATURES].mean()) / table[FEATURES].std(ddof=0)
    tables.append(table)

    for when, s in [("before", stages), ("after", stages[start:stop])]:
        counts.append({"recording": row.recording, "when": when,
                       **pd.Series(s).value_counts().to_dict()})

features = pd.concat(tables, ignore_index=True)
columns = helpers.STAGES + ["unscored"]
counts = pd.DataFrame(counts)
counts[columns] = counts[columns].fillna(0).astype(int)

assert np.isfinite(features[FEATURES].to_numpy()).all()
per_recording = features.groupby("recording")[FEATURES]
print(f"{len(features):,} epochs, {features['recording'].nunique()} recordings, "
      f"{features['subject'].nunique()} subjects")
print("Largest |mean|:", per_recording.mean().abs().max().max().round(6))
print("Std range:", per_recording.std(ddof=0).min().min().round(6),
      "to", per_recording.std(ddof=0).max().max().round(6))
counts.groupby("when")[columns].sum().loc[["before", "after"]]

170,252 epochs, 153 recordings, 78 subjects
Largest |mean|: 0.0
Std range: 1.0 to 1.0


,W,N1,N2,N3,REM,unscored
when,,,,,,
before,285433,21522,69132,13039,25835,1463
after,42563,20808,68156,12923,25802,297


The result is 170,252 scored epochs from 153 recordings and 78 subjects, with mean 0 and standard deviation 1 in every recording. Trimming cuts wake from 285,433 to 42,563 epochs, which is 68.8% down to 25.0% of scored epochs. It drops 1,839 sleep epochs: 976 of N2, 714 of N1, 116 of N3, and 33 of REM. The 297 unscored epochs left inside the windows are removed.

---

### Checking the trimming rule

The three-hour gap is our choice, so we compare it with other gaps and with keeping all sleep blocks. A good rule should remove most of the daytime wake without removing much sleep.

In [62]:
RULES = {"all sleep blocks": np.inf, "1 h gap": 120, "2 h gap": 240,
         "3 h gap (used)": MAX_GAP, "4 h gap": 480}
n_sleep = sum(np.isin(s, helpers.STAGES[1:]).sum() for s in all_stages.values())

rows = []
for name, max_gap in RULES.items():
    kept = [s[slice(*trim_bounds(s, max_gap))] for s in all_stages.values()]
    hours = np.array([len(s) for s in kept]) * helpers.EPOCH_SEC / 3600
    scored = np.concatenate(kept)
    scored = scored[scored != "unscored"]
    rows.append({"rule": name, "epochs": len(scored),
                 "wake_pct": 100 * (scored == "W").mean(),
                 "sleep_dropped": n_sleep - (scored != "W").sum(),
                 "sleep_dropped_pct": 100 * (1 - (scored != "W").sum() / n_sleep),
                 "over_12h": (hours > 12).sum(), "max_hours": hours.max()})
pd.DataFrame(rows).set_index("rule").round(1)

,epochs,wake_pct,sleep_dropped,sleep_dropped_pct,over_12h,max_hours
rule,,,,,,
all sleep blocks,195469,33.7,0,0.0,40,23.4
1 h gap,161028,22.1,4140,3.2,3,12.8
2 h gap,166966,24.0,2619,2.0,6,19.3
3 h gap (used),170252,25.0,1839,1.4,8,19.3
4 h gap,171909,25.7,1754,1.4,10,19.3


Keeping all sleep blocks leaves 33.7% wake and 40 windows longer than 12 hours, and every gap rule brings that down to 10 windows or fewer. Shorter gaps remove more wake but also more sleep: a one-hour gap drops 3.2% of sleep epochs, two hours drops 2.0%, and three hours drops 1.4%. **Beyond three hours there is little left to gain:** four hours keeps only 85 more sleep epochs and adds wake, so we use three hours. Eight windows are still longer than 12 hours.

---

### Building a label-free window for a sensitivity check

The main trimming rule finds the sleep period from the expert hypnograms. To check whether that affects our results, we build a second window that uses no stage labels to set its bounds. It starts 30 minutes before lights-off and ends 9 hours after, with the same rule for every recording. The lights-off times come from `SC-subjects.xls`, part of the same PhysioNet dataset (saved as `data/sc_subjects.csv`). Features are z-scored over the whole window, so the normalization uses no labels either.

The spreadsheet has no wake-up time, so the window length is a trade-off: a shorter window cuts off the end of some nights, and a longer one adds wake. Nine hours is a compromise that we chose with the labels in view, but it is the same for everyone.

In [63]:
lights = pd.read_csv(os.path.join(helpers.BASE, "data", "sc_subjects.csv"), dtype={"subject": str})
lights = lights.set_index(["subject", "night"])["lights_off"]

def lights_off_bounds(row, n_epochs, hours_before=0.5, hours_after=9):
    began = mne.io.read_raw_edf(row.psg_path, verbose="error").info["meas_date"]
    hour, minute = map(int, lights[row.subject, row.night].split(":"))
    minutes_until_off = (hour * 60 + minute - began.hour * 60 - began.minute) % (24 * 60)
    off = minutes_until_off * 60 // helpers.EPOCH_SEC
    start = max(0, off - int(hours_before * 3600 / helpers.EPOCH_SEC))
    stop  = min(n_epochs, off + int(hours_after * 3600 / helpers.EPOCH_SEC))
    return start, stop

tables = []
for row in files.itertuples():
    signals, stages, sfreq = load_epochs(row)
    start, stop = lights_off_bounds(row, len(stages))

    table = compute_features({c: x[start:stop] for c, x in signals.items()}, sfreq)
    table = (table - table.mean()) / table.std(ddof=0)
    table.insert(0, "recording", row.recording)
    table.insert(1, "subject", row.subject)
    table.insert(2, "night", row.night)
    table.insert(3, "epoch", np.arange(start, stop))
    table.insert(4, "stage", stages[start:stop])
    tables.append(table[table["stage"] != "unscored"])

lights_off = pd.concat(tables, ignore_index=True)
assert np.isfinite(lights_off[FEATURES].to_numpy()).all()
lights_off.to_csv(os.path.join(helpers.PROC, "features_lights_off.csv.gz"), index=False, float_format="%.6f")

main_sleep = features[features["stage"] != "W"]
kept = main_sleep.merge(lights_off[["recording", "epoch"]], on=["recording", "epoch"])
print(f"{len(lights_off):,} epochs, {(lights_off['stage'] == 'W').mean():.1%} wake, "
      f"keeps {len(kept) / len(main_sleep):.1%} of the sleep in the main sleep period")

174,007 epochs, 28.6% wake, keeps 97.3% of the sleep in the main sleep period


The label-free window has 174,007 epochs, 28.6% of them wake, compared with 25.0% under the main rule. It keeps 97.3% of the sleep in the main sleep period, so it covers nearly the same sleep **without using any stage label to set the window**.

---

### Saving outputs

`features.csv.gz` has one row per scored epoch, with its recording, subject, night, epoch number, stage, and seven features. `stage_counts.csv` has the stage counts of each recording before and after trimming. The epoch number is the position in the full recording, so a jump of more than one marks a gap where a sequence must be split.

In [64]:
features_path = os.path.join(helpers.PROC, "features.csv.gz")
counts_path   = os.path.join(helpers.PROC, "stage_counts.csv")

features.to_csv(features_path, index=False, float_format="%.6f")
counts.to_csv(counts_path, index=False)

gaps = int((features.groupby("recording")["epoch"].diff() > 1).sum())
print(f"Saved {len(features):,} rows to features.csv.gz")
print(f"{gaps} gaps, {gaps + features['recording'].nunique()} sequences")

Saved 170,252 rows to features.csv.gz
140 gaps, 293 sequences


We save 170,252 rows. There are 140 gaps left by unscored epochs, so the 153 recordings become 293 sequences. The HMM treats each one as separate and never counts a transition across a gap.

---